## 2. Demography Cleaning Pipeline
Load `demography.csv`, inspect quality, clean critical fields, validate measurements, recompute BMI, run QA checks, and save `demography_clean.csv`.

In [ ]:
# 2.1 Load and inspect demography data
dfDEMO = pd.read_csv("data_files/demography.csv")
dfDEMO_original = dfDEMO.copy()

print("Shape:", dfDEMO.shape)
display(dfDEMO.head())
dfDEMO.info()

print("\nMissing values per column:")
display(dfDEMO.isna().sum().sort_values(ascending=False))

print("\nDuplicate inpatient_number count:", dfDEMO["inpatient_number"].duplicated().sum())

In [ ]:
# 2.2 Drop rows where all critical demographic fields are missing
critical_demo_cols = ["gender", "weight", "height", "occupation", "agecat"]
dfDEMO = dfDEMO.dropna(subset=critical_demo_cols, how="all").copy()

# 2.3 Rename columns for consistency
dfDEMO = dfDEMO.rename(columns={
    "inpatient_number": "patient_id",
    "agecat": "age_category"
})

# 2.4 Standardize text columns
text_cols = ["gender", "occupation", "age_category"]
for col in text_cols:
    dfDEMO[col] = dfDEMO[col].astype("string").str.strip()

dfDEMO["occupation"] = dfDEMO["occupation"].replace({
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "worker": "Worker"
})
dfDEMO["occupation"] = dfDEMO["occupation"].fillna("Unknown")

# 2.5 Flag and null invalid measurements
dfDEMO["measurement_issue"] = False
invalid_weight = dfDEMO["weight"] <= 0
invalid_height = dfDEMO["height"] < 1.0

dfDEMO.loc[invalid_weight | invalid_height, "measurement_issue"] = True
dfDEMO.loc[invalid_weight, "weight"] = np.nan
dfDEMO.loc[invalid_height, "height"] = np.nan

# 2.6 Recompute BMI from cleaned height/weight
dfDEMO["bmi_original"] = pd.to_numeric(dfDEMO["bmi"], errors="coerce").round(2)
dfDEMO["bmi"] = (dfDEMO["weight"] / (dfDEMO["height"] ** 2)).round(2)

# 2.7 Enforce plausible BMI range
dfDEMO["bmi"] = pd.to_numeric(dfDEMO["bmi"], errors="coerce")
dfDEMO.loc[(dfDEMO["bmi"] < 15) | (dfDEMO["bmi"] > 60), "bmi"] = np.nan

In [ ]:
# 2.8 Post-clean validation checks
print("Rows after cleaning:", len(dfDEMO))
print("Unique patient IDs:", dfDEMO["patient_id"].nunique())
print("Duplicate patient IDs:", dfDEMO["patient_id"].duplicated().sum())

print("\nMissing values after cleaning:")
display(dfDEMO.isna().sum().sort_values(ascending=False))

print("\nNumeric summary (weight, height, bmi):")
display(dfDEMO[["weight", "height", "bmi"]].describe())

print("\nCategory distributions:")
for c in ["gender", "occupation", "age_category"]:
    print(f"\n{c}")
    print(dfDEMO[c].value_counts(dropna=False))

print("\nRecords flagged for measurement_issue:")
display(dfDEMO.loc[dfDEMO["measurement_issue"], ["patient_id", "weight", "height", "bmi_original"]])

# 2.9 Save cleaned output
dfDEMO.to_csv("cleaned_files/demography_clean.csv", index=False)
print("Saved cleaned_files/demography_clean.csv")